# BioJEPA v0.6 Data Prep - Notebook 3: Shard Generation

This notebook handles:
1. Processing each dataset incrementally (chunked for large ones)
2. Normalizing expression (CPM + log1p)
3. Building control banks per batch/gem_group
4. Writing pretraining shards (train/val/test splits respecting perturbation boundaries)
5. Writing training shards with multi-pert format (skips perturbations missing both embeddings)

**Pretraining split strategy:**
- Control cells: randomly split to train/val/test following VAL_PCT and TEST_PCT
- Perturbed cells: assigned to same split as their perturbation (train/val/test)
- This ensures no test perturbation expression patterns leak into pretraining train

**Inputs (from Notebooks 1 & 2):**
- `gene_to_idx.json`, `gene_names.json` - gene universe
- `dataset_gene_masks.json` - per-dataset gene masks
- `dataset_splits.json` - train/val/test perturbation sets
- `holdout_perturbations.json` - test split genes to exclude
- `pert_embd/seq_banks/dna_to_idx.json` - UNIFIED DNA embeddings mapping
- `pert_embd/seq_banks/chemical_to_idx.json` - UNIFIED chemical embeddings mapping
- `pert_embd/target_banks/gene_to_target_idx.json` - protein target mapping
- `pert_embd/{train,val,test}/align_*.npz` - alignment pairs (from Notebook 2)
- `skipped_perturbations.json` - perturbations to skip (missing both sequence AND target)

**Outputs:**
- `pretraining/{train,val,test}/pt_*.npz`
- `training/{train,val,test}/shard_*.npz`

In [1]:
from pathlib import Path
from scipy.sparse import issparse
from tqdm import tqdm
import pandas as pd
import numpy as np
import scanpy as sc
import json
import gc

In [2]:
ref_dir = Path('/Users/djemec/data/jepa/reference_data')
data_dir = Path('/Users/djemec/data/jepa/v0_6')

pretraining_dir = data_dir / 'pretraining'
training_dir = data_dir / 'training'
pert_embd_dir = data_dir / 'pert_embd'

for split in ['train', 'val', 'test']:
    (pretraining_dir / split).mkdir(parents=True, exist_ok=True)
    (training_dir / split).mkdir(parents=True, exist_ok=True)
    (pert_embd_dir / split).mkdir(parents=True, exist_ok=True)

In [3]:
CHUNK_SIZE = 20000
SHARD_SIZE = 2560
PT_SHARD_SIZE = 2560
COUNT_NORMALIZE_TARGET = 1e4
MAX_N_PERT = 4
MIN_GENES_PER_CELL = 50
VAL_PCT = 0.05
TEST_PCT = 0.10
SEED = 42
np.random.seed(SEED)

## Load Metadata from Previous Notebooks

In [4]:
with open(data_dir / 'gene_to_idx.json') as f:
    gene_to_idx = json.load(f)

with open(data_dir / 'gene_names.json') as f:
    gene_names = json.load(f)

with open(data_dir / 'dataset_gene_masks.json') as f:
    dataset_gene_masks = {k: np.array(v, dtype=bool) for k, v in json.load(f).items()}

with open(data_dir / 'dataset_splits.json') as f:
    dataset_splits = json.load(f)

with open(data_dir / 'holdout_perturbations.json') as f:
    holdout_perts = set(json.load(f))

with open(data_dir / 'cell_type_to_id.json') as f:
    cell_type_to_id = json.load(f)

N_GENES = len(gene_to_idx)
print(f'Gene universe: {N_GENES}')
print(f'Holdout perturbations: {len(holdout_perts)}')

Gene universe: 10000
Holdout perturbations: 1023


In [5]:
seq_banks = pert_embd_dir / 'seq_banks'
target_banks = pert_embd_dir / 'target_banks'

with open(seq_banks / 'dna_to_idx.json') as f:
    dna_to_idx = json.load(f)

with open(seq_banks / 'chemical_to_idx.json') as f:
    chemical_to_idx = json.load(f)

with open(target_banks / 'gene_to_target_idx.json') as f:
    gene_to_target_idx = json.load(f)

print(f'DNA embeddings: {len(dna_to_idx)} (unified: CRISPRi sgID_AB + Adamson perts + Norman guide_ids)')
print(f'Chemical embeddings: {len(chemical_to_idx)} (unified: SMILES + drug names)')
print(f'Target embeddings: {len(gene_to_target_idx)}')

skip_file = data_dir / 'skipped_perturbations.json'
if skip_file.exists():
    with open(skip_file) as f:
        skip_sets = json.load(f)
    crispri_skip_sgids = set(skip_sets.get('crispri_sgids', []))
    adamson_skip_perts = set(skip_sets.get('adamson_perts', []))
    norman_skip_guides = set(skip_sets.get('norman_guides', []))
    sciplex_skip_drugs = set(skip_sets.get('sciplex_drugs', []))
    print(f'\nWill skip for training shards (missing BOTH sequence AND target):')
    print(f'  CRISPRi: {len(crispri_skip_sgids)}, Adamson: {len(adamson_skip_perts)}, Norman: {len(norman_skip_guides)}, Sciplex: {len(sciplex_skip_drugs)}')
else:
    crispri_skip_sgids = set()
    adamson_skip_perts = set()
    norman_skip_guides = set()
    sciplex_skip_drugs = set()
    print(f'\nWarning: skipped_perturbations.json not found - no perturbations will be skipped')

DNA embeddings: 11643 (unified: CRISPRi sgID_AB + Adamson perts + Norman guide_ids)
Chemical embeddings: 376 (unified: SMILES + drug names)
Target embeddings: 19567

Will skip for training shards (missing BOTH sequence AND target):
  CRISPRi: 0, Adamson: 1, Norman: 3, Sciplex: 0


In [6]:
datasets = {
    'k562e_raw': ref_dir / 'raw_k562e' / 'K562_essential_raw_singlecell_01.h5ad',
    'rep1e': ref_dir / 'rep1e' / 'rpe1_raw_singlecell_01.h5ad',
    'k562gw': ref_dir / 'k562gw' / 'K562_gwps_raw_singlecell_01.h5ad',
    'adamson': ref_dir / 'adamson' / 'AdamsonWeissman2016_GSM2406681_10X010.h5ad',
    'norman': ref_dir / 'norman' / 'NormanWeissman2019_filtered.h5ad',
    'sciplex': ref_dir / 'sciplex' / 'SrivatsanTrapnell2020_sciplex3.h5ad',
}

## Helper Functions

In [7]:
def is_valid(val):
    if pd.isna(val):
        return False
    if isinstance(val, str) and (val.strip() == '' or val.strip().lower() == 'nan'):
        return False
    return True

def clean_gears_name(name):
    if not is_valid(name):
        return None
    name = str(name)
    if name.endswith('+ctrl'):
        return name.replace('+ctrl', '')
    if name.startswith('ctrl+'):
        return name.replace('ctrl+', '')
    if name.lower() == 'ctrl':
        return 'control'
    return name.strip()

In [8]:
def normalize_chunk(X, total_counts=None):
    if total_counts is None:
        total_counts = X.sum(axis=1, keepdims=True)
    total_counts = np.maximum(total_counts, 1)
    X_norm = X / total_counts * COUNT_NORMALIZE_TARGET
    return np.log1p(X_norm)

def map_to_universe(expr_row, dataset_mask, dataset_gene_indices):
    out = np.zeros(N_GENES, dtype=np.float32)
    out[dataset_mask] = expr_row[dataset_gene_indices]
    return out

In [9]:
def get_dataset_gene_mapping(adata, gene_to_idx):
    var_df = adata.var
    
    if var_df.index[0].startswith('ENSG'):
        ensg_col = var_df.index
    elif 'ensembl_id' in var_df.columns:
        ensg_col = var_df['ensembl_id']
    elif 'ensemble_id' in var_df.columns:
        ensg_col = var_df['ensemble_id']
    else:
        ensg_col = var_df.index
    
    ds_gene_to_local_idx = {}
    for i, ensg in enumerate(ensg_col):
        if is_valid(ensg) and ensg in gene_to_idx:
            ds_gene_to_local_idx[ensg] = i
    
    universe_mask = np.zeros(N_GENES, dtype=bool)
    local_indices = []
    
    for ensg, local_idx in ds_gene_to_local_idx.items():
        universe_idx = gene_to_idx[ensg]
        universe_mask[universe_idx] = True
        local_indices.append(local_idx)
    
    local_indices = np.array(local_indices)
    return universe_mask, local_indices, ds_gene_to_local_idx

In [10]:
def build_control_bank(adata, batch_col, condition_col, universe_mask, local_indices):
    control_bank = {}
    
    if condition_col not in adata.obs.columns:
        print(f'  Warning: {condition_col} not in obs, trying alternatives')
        for alt in ['perturbation', 'gene', 'product_name']:
            if alt in adata.obs.columns:
                condition_col = alt
                break
    
    obs = adata.obs
    ctrl_keywords = ['ctrl', 'control', 'vehicle', 'dmso', 'non-targeting','*','negative_control','nan','negctrl0','negctrl']
    ctrl_mask = obs[condition_col].astype(str).str.lower().isin(ctrl_keywords)
    
    if batch_col not in obs.columns:
        batch_col = 'gem_group' if 'gem_group' in obs.columns else None
    
    if batch_col is None:
        batches = ['all']
        batch_values = pd.Series(['all'] * len(obs))
    else:
        batches = obs[batch_col].unique()
        batch_values = obs[batch_col]
    
    total_filtered = 0
    for batch in batches:
        if batch_col is None:
            batch_ctrl = ctrl_mask
        else:
            batch_ctrl = ctrl_mask & (batch_values == batch)
        
        indices = np.where(batch_ctrl)[0]
        if len(indices) == 0:
            continue
        
        X_ctrl = adata.X[indices]
        if issparse(X_ctrl):
            X_ctrl = X_ctrl.toarray()
        
        gene_counts = (X_ctrl > 0).sum(axis=1)
        valid_mask = gene_counts >= MIN_GENES_PER_CELL
        n_filtered = len(indices) - valid_mask.sum()
        total_filtered += n_filtered
        X_ctrl = X_ctrl[valid_mask]
        indices = indices[valid_mask]
        
        if len(X_ctrl) == 0:
            continue
        
        totals = X_ctrl.sum(axis=1)
        X_norm = normalize_chunk(X_ctrl, totals.reshape(-1, 1))
        
        X_mapped = np.zeros((len(indices), N_GENES), dtype=np.float32)
        X_mapped[:, universe_mask] = X_norm[:, local_indices]
        
        control_bank[batch] = {
            'X': X_mapped,
            'total': np.log1p(totals).astype(np.float32)
        }
    
    print(f'  Built control bank with {len(control_bank)} batches, {sum(len(v["X"]) for v in control_bank.values())} cells (filtered {total_filtered} low-gene-count)')
    return control_bank, batch_col, condition_col

In [11]:
def write_training_shard(buffer, shard_idx, ds_name, split, gene_mask):
    path = training_dir / split / f'shard_{ds_name}_{split}_{shard_idx:04d}.npz'
    
    np.savez_compressed(path,
        control=np.array(buffer['control'], dtype=np.float32),
        control_total=np.array(buffer['control_total'], dtype=np.float32),
        case=np.array(buffer['case'], dtype=np.float32),
        case_total=np.array(buffer['case_total'], dtype=np.float32),
        seq_idx=np.array(buffer['seq_idx'], dtype=np.int32),
        target_idx=np.array(buffer['target_idx'], dtype=np.int32),
        modality=np.array(buffer['modality'], dtype=np.int8),
        mode=np.array(buffer['mode'], dtype=np.int8),
        has_seq=np.array(buffer['has_seq'], dtype=np.bool_),
        has_target=np.array(buffer['has_target'], dtype=np.bool_),
        n_perts=np.array(buffer['n_perts'], dtype=np.int8),
        dose=np.array(buffer['dose'], dtype=np.float32),
        batch_id=np.array(buffer['batch_id'], dtype=np.int32),
        cell_type=np.array(buffer['cell_type'], dtype=np.int8),
        gene_mask=gene_mask.astype(np.bool_)
    )
    return path

def write_pretraining_shard(X, totals, gene_mask, shard_idx, ds_name, split):
    path = pretraining_dir / split / f'pt_{ds_name}_{split}_{shard_idx:04d}.npz'
    np.savez_compressed(path,
        x=X.astype(np.float32),
        total=totals.astype(np.float32),
        gene_mask=gene_mask.astype(np.bool_)
    )
    return path

In [12]:
def empty_buffer():
    return {
        'control': [], 'control_total': [], 'case': [], 'case_total': [],
        'seq_idx': [], 'target_idx': [], 'modality': [], 'mode': [],
        'has_seq': [], 'has_target': [], 'n_perts': [], 'dose': [],
        'batch_id': [], 'cell_type': []
    }

## Process k562e_raw (Primary Dataset)

In [13]:
ds_name = 'k562e_raw'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'Shape: {adata.shape}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'Genes in universe: {universe_mask.sum()}')

splits = dataset_splits[ds_name]
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

Processing k562e_raw...
Shape: (310385, 8563)
Genes in universe: 8563


In [14]:
control_bank, batch_col, condition_col = build_control_bank(
    adata, 'gem_group', 'gene', universe_mask, local_indices
)

  Built control bank with 48 batches, 10691 cells (filtered 0 low-gene-count)


In [15]:
def get_split_for_pert(pert, train_perts, val_perts, test_perts):
    if pert in train_perts:
        return 'train'
    elif pert in val_perts:
        return 'val'
    elif pert in test_perts:
        return 'test'
    return None

In [16]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': [], 'test': []}
pt_totals = {'train': [], 'val': [], 'test': []}
pt_shard_counts = {'train': 0, 'val': 0, 'test': 0}
skipped_count = 0
low_gene_count = 0
unassigned_pert_count = 0

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    chunk_gene_counts = (chunk_X > 0).sum(axis=1)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        if chunk_gene_counts[i] < MIN_GENES_PER_CELL:
            low_gene_count += 1
            continue
        
        pert = clean_gears_name(row.get('gene', row.get('condition', '')))
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = pert is None or pert == 'control'
        
        if is_control:
            r = np.random.random()
            if r < TEST_PCT:
                pt_split = 'test'
            elif r < TEST_PCT + VAL_PCT:
                pt_split = 'val'
            else:
                pt_split = 'train'
        else:
            pt_split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        
        if pt_split is not None:
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        else:
            if not is_control:
                unassigned_pert_count += 1
        
        if is_control:
            continue
        
        sgid_ab = str(row.get('sgID_AB', '')).replace(',', '-').strip()
        if sgid_ab in crispri_skip_sgids:
            skipped_count += 1
            continue
        
        split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get('gem_group', 'all')
        if batch not in control_bank:
            continue
        
        seq_idx = dna_to_idx.get(sgid_ab, -1)
        target_idx = gene_to_target_idx.get(pert, gene_to_target_idx.get(row.get('gene_id', ''), -1))
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        buf['seq_idx'].append([seq_idx, -1, -1, -1])
        buf['target_idx'].append([target_idx, -1, -1, -1])
        buf['modality'].append([0, -1, -1, -1])
        buf['mode'].append([0, -1, -1, -1])
        buf['has_seq'].append([seq_idx >= 0, False, False, False])
        buf['has_target'].append([target_idx >= 0, False, False, False])
        buf['n_perts'].append(1)
        buf['dose'].append([-1.0, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_to_id.get('K562', 0))
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs, chunk_gene_counts
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val', 'test']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]}, test={pt_shard_counts["test"]} shards')
print(f'  Skipped {skipped_count} cells (missing embeddings), {low_gene_count} cells (low gene count), {unassigned_pert_count} cells (unassigned pert)')

k562e_raw: 100%|██████████████████████████████████████████████████████████████████| 16/16 [08:05<00:00, 30.36s/it]


k562e_raw complete: train=48, val=6, test=19 shards
  Pretraining: train=48, val=6, test=19 shards
  Skipped 0 cells (missing embeddings), 0 cells (low gene count), 128957 cells (unassigned pert)


In [17]:
del adata, control_bank
gc.collect()

0

## Process rep1e Dataset (CRISPRi, Large)

In [18]:
ds_name = 'rep1e'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'Shape: {adata.shape}')
print(f'Obs columns: {list(adata.obs.columns)[:15]}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'Genes in universe: {universe_mask.sum()}')

splits = dataset_splits.get(ds_name, {'train': [], 'val': [], 'test': []})
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

Processing rep1e...
Shape: (247914, 8749)
Obs columns: ['gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count']
Genes in universe: 8162


In [19]:
control_bank, batch_col, condition_col = build_control_bank(
    adata, 'gem_group', 'gene', universe_mask, local_indices
)

  Built control bank with 56 batches, 11485 cells (filtered 0 low-gene-count)


In [20]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': [], 'test': []}
pt_totals = {'train': [], 'val': [], 'test': []}
pt_shard_counts = {'train': 0, 'val': 0, 'test': 0}
skipped_count = 0
low_gene_count = 0
unassigned_pert_count = 0

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    chunk_gene_counts = (chunk_X > 0).sum(axis=1)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        if chunk_gene_counts[i] < MIN_GENES_PER_CELL:
            low_gene_count += 1
            continue
        
        pert = clean_gears_name(row.get('gene', row.get('condition', '')))
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = pert is None or pert == 'control'
        
        if is_control:
            r = np.random.random()
            if r < TEST_PCT:
                pt_split = 'test'
            elif r < TEST_PCT + VAL_PCT:
                pt_split = 'val'
            else:
                pt_split = 'train'
        else:
            pt_split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        
        if pt_split is not None:
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        else:
            if not is_control:
                unassigned_pert_count += 1
        
        if is_control:
            continue
        
        sgid_ab = str(row.get('sgID_AB', '')).replace(',', '-').strip()
        if sgid_ab in crispri_skip_sgids:
            skipped_count += 1
            continue
        
        split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get('gem_group', 'all')
        if batch not in control_bank:
            continue
        
        seq_idx = dna_to_idx.get(sgid_ab, -1)
        target_idx = gene_to_target_idx.get(pert, gene_to_target_idx.get(row.get('gene_id', ''), -1))
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        buf['seq_idx'].append([seq_idx, -1, -1, -1])
        buf['target_idx'].append([target_idx, -1, -1, -1])
        buf['modality'].append([0, -1, -1, -1])
        buf['mode'].append([0, -1, -1, -1])
        buf['has_seq'].append([seq_idx >= 0, False, False, False])
        buf['has_target'].append([target_idx >= 0, False, False, False])
        buf['n_perts'].append(1)
        buf['dose'].append([-1.0, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_to_id.get('RPE1', 1))
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs, chunk_gene_counts
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val', 'test']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]}, test={pt_shard_counts["test"]} shards')
print(f'  Skipped {skipped_count} cells (missing embeddings), {low_gene_count} cells (low gene count), {unassigned_pert_count} cells (unassigned pert)')

rep1e: 100%|██████████████████████████████████████████████████████████████████████| 13/13 [10:23<00:00, 47.94s/it]


rep1e complete: train=80, val=5, test=9 shards
  Pretraining: train=80, val=5, test=9 shards
  Skipped 0 cells (missing embeddings), 0 cells (low gene count), 11485 cells (unassigned pert)


In [21]:
del adata, control_bank
gc.collect()

0

## Process k562gw Dataset (CRISPRi, Genome-Wide)

In [22]:
ds_name = 'k562gw'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'  Shape: {adata.shape}')
print(f'  Obs columns: {list(adata.obs.columns)[:15]}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'  Genes in universe: {universe_mask.sum()}')

splits = dataset_splits.get(ds_name, {'train': [], 'val': [], 'test': []})
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

Processing k562gw...
  Shape: (1989578, 8248)
  Obs columns: ['gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count']
  Genes in universe: 8248


In [23]:
control_bank, batch_col, condition_col = build_control_bank(
    adata, 'gem_group', 'gene', universe_mask, local_indices
)

  Built control bank with 267 batches, 75328 cells (filtered 0 low-gene-count)


In [24]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': [], 'test': []}
pt_totals = {'train': [], 'val': [], 'test': []}
pt_shard_counts = {'train': 0, 'val': 0, 'test': 0}
skipped_count = 0
low_gene_count = 0
unassigned_pert_count = 0

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    chunk_gene_counts = (chunk_X > 0).sum(axis=1)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        if chunk_gene_counts[i] < MIN_GENES_PER_CELL:
            low_gene_count += 1
            continue
        
        pert_raw = row.get('gene', row.get('condition', ''))
        pert = clean_gears_name(pert_raw)
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = pert is None or pert == 'control' or str(pert_raw).lower() == 'non-targeting'
        
        if is_control:
            r = np.random.random()
            if r < TEST_PCT:
                pt_split = 'test'
            elif r < TEST_PCT + VAL_PCT:
                pt_split = 'val'
            else:
                pt_split = 'train'
        else:
            pt_split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        
        if pt_split is not None:
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        else:
            if not is_control:
                unassigned_pert_count += 1
        
        if is_control:
            continue
        
        sgid_ab = str(row.get('sgID_AB', '')).replace(',', '-').strip()
        if sgid_ab in crispri_skip_sgids:
            skipped_count += 1
            continue
        
        split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get('gem_group', 'all')
        if batch not in control_bank:
            continue
        
        seq_idx = dna_to_idx.get(sgid_ab, -1)
        gene_id = row.get('gene_id', '')
        target_idx = gene_to_target_idx.get(pert, gene_to_target_idx.get(gene_id, -1))
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        buf['seq_idx'].append([seq_idx, -1, -1, -1])
        buf['target_idx'].append([target_idx, -1, -1, -1])
        buf['modality'].append([0, -1, -1, -1])
        buf['mode'].append([0, -1, -1, -1])
        buf['has_seq'].append([seq_idx >= 0, False, False, False])
        buf['has_target'].append([target_idx >= 0, False, False, False])
        buf['n_perts'].append(1)
        buf['dose'].append([-1.0, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_to_id.get('K562', 0))
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs, chunk_gene_counts
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val', 'test']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]}, test={pt_shard_counts["test"]} shards')
print(f'  Skipped {skipped_count} cells (missing embeddings), {low_gene_count} cells (low gene count), {unassigned_pert_count} cells (unassigned pert)')

k562gw: 100%|█████████████████████████████████████████████████████████████████| 100/100 [1:20:03<00:00, 48.04s/it]


k562gw complete: train=640, val=38, test=72 shards
  Pretraining: train=665, val=39, test=75 shards
  Skipped 0 cells (missing embeddings), 0 cells (low gene count), 0 cells (unassigned pert)


In [25]:
del adata, control_bank
gc.collect()

0

## Process Adamson Dataset

In [26]:
ds_name = 'adamson'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'  Shape: {adata.shape}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'  Genes in universe: {universe_mask.sum()}')

splits = dataset_splits.get(ds_name, {'train': [], 'val': [], 'test': []})
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

Processing adamson...
  Shape: (65337, 32738)
  Genes in universe: 9749


In [27]:
control_bank, batch_col, condition_col = build_control_bank(
    adata, 'gem_group', 'perturbation', universe_mask, local_indices
)

  Built control bank with 1 batches, 2714 cells (filtered 0 low-gene-count)


In [28]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': [], 'test': []}
pt_totals = {'train': [], 'val': [], 'test': []}
pt_shard_counts = {'train': 0, 'val': 0, 'test': 0}
skipped_count = 0
low_gene_count = 0
unassigned_pert_count = 0

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    chunk_gene_counts = (chunk_X > 0).sum(axis=1)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        if chunk_gene_counts[i] < MIN_GENES_PER_CELL:
            low_gene_count += 1
            continue
        
        pert = str(row.get('perturbation', row.get('condition', ''))).strip()
        pert_clean = pert.split('/')[0].strip() if '/' in pert else pert
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = pert.lower() in ['control', 'ctrl', 'nan', '']
        
        if is_control:
            r = np.random.random()
            if r < TEST_PCT:
                pt_split = 'test'
            elif r < TEST_PCT + VAL_PCT:
                pt_split = 'val'
            else:
                pt_split = 'train'
        else:
            pt_split = get_split_for_pert(pert_clean, train_perts, val_perts, test_perts)
            if pt_split is None:
                pt_split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        
        if pt_split is not None:
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        else:
            if not is_control:
                unassigned_pert_count += 1
        
        if is_control:
            continue
        
        if pert in adamson_skip_perts:
            skipped_count += 1
            continue
        
        split = get_split_for_pert(pert_clean, train_perts, val_perts, test_perts)
        if split is None:
            split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get(batch_col, 'all') if batch_col else 'all'
        if batch not in control_bank:
            batch = list(control_bank.keys())[0] if control_bank else None
        if batch is None:
            continue
        
        seq_idx = dna_to_idx.get(pert, dna_to_idx.get(pert_clean, -1))
        target_idx = gene_to_target_idx.get(pert_clean, gene_to_target_idx.get(pert, -1))
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        buf['seq_idx'].append([seq_idx, -1, -1, -1])
        buf['target_idx'].append([target_idx, -1, -1, -1])
        buf['modality'].append([0, -1, -1, -1])
        buf['mode'].append([0, -1, -1, -1])
        buf['has_seq'].append([seq_idx >= 0, False, False, False])
        buf['has_target'].append([target_idx >= 0, False, False, False])
        buf['n_perts'].append(1)
        buf['dose'].append([-1.0, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_to_id.get('K562', 0))
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs, chunk_gene_counts
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val', 'test']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]}, test={pt_shard_counts["test"]} shards')
print(f'  Skipped {skipped_count} cells (missing embeddings), {low_gene_count} cells (low gene count), {unassigned_pert_count} cells (unassigned pert)')

adamson: 100%|██████████████████████████████████████████████████████████████████████| 4/4 [02:57<00:00, 44.40s/it]


adamson complete: train=19, val=4, test=3 shards
  Pretraining: train=20, val=4, test=3 shards
  Skipped 1283 cells (missing embeddings), 0 cells (low gene count), 0 cells (unassigned pert)


In [29]:
del adata, control_bank
gc.collect()

0

## Process Norman Dataset (CRISPRa, dual-gene)

In [30]:
ds_name = 'norman'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'  Shape: {adata.shape}')
print(f'  Obs columns: {adata.obs.columns.tolist()}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'  Genes in universe: {universe_mask.sum()}')

splits = dataset_splits.get(ds_name, {'train': [], 'val': [], 'test': []})
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

Processing norman...
  Shape: (111445, 33694)
  Obs columns: ['guide_id', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo']
  Genes in universe: 9865


In [31]:
norman_target_df = pd.read_csv(ref_dir / 'norman' / 'norman_guideid_ensemble_id_map.csv')

guide_to_ensg = {}
for _, row in norman_target_df.iterrows():
    guide_id = row['guide_id']
    if is_valid(guide_id):
        first_ensg = row.get('first_id')
        second_ensg = row.get('second_id')
        guide_to_ensg[guide_id] = (first_ensg if is_valid(first_ensg) else None,
                                   second_ensg if is_valid(second_ensg) else None)

In [32]:
control_bank, batch_col, condition_col = build_control_bank(
    adata, 'gem_group', 'guide_id', universe_mask, local_indices
)

  Built control bank with 0 batches, 0 cells (filtered 0 low-gene-count)


In [33]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': [], 'test': []}
pt_totals = {'train': [], 'val': [], 'test': []}
pt_shard_counts = {'train': 0, 'val': 0, 'test': 0}
skipped_count = 0
low_gene_count = 0
unassigned_pert_count = 0

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    chunk_gene_counts = (chunk_X > 0).sum(axis=1)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        if chunk_gene_counts[i] < MIN_GENES_PER_CELL:
            low_gene_count += 1
            continue
        
        guide_id = str(row.get('guide_id', '')).strip()
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = guide_id.lower() in ['control', 'ctrl', 'nan', ''] or 'negctrl' in guide_id.lower()
        
        parts = guide_id.split('_')
        gene_a = parts[0] if len(parts) >= 1 else None
        gene_b = parts[1] if len(parts) >= 2 and parts[1].lower() != 'negctrl0' else None
        
        if is_control:
            r = np.random.random()
            if r < TEST_PCT:
                pt_split = 'test'
            elif r < TEST_PCT + VAL_PCT:
                pt_split = 'val'
            else:
                pt_split = 'train'
        else:
            pt_split = get_split_for_pert(guide_id, train_perts, val_perts, test_perts)
            if pt_split is None and gene_a:
                pt_split = get_split_for_pert(gene_a, train_perts, val_perts, test_perts)
        
        if pt_split is not None:
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        else:
            if not is_control:
                unassigned_pert_count += 1
        
        if is_control:
            continue
        
        if guide_id in norman_skip_guides:
            skipped_count += 1
            continue
        
        split = get_split_for_pert(guide_id, train_perts, val_perts, test_perts)
        if split is None and gene_a:
            split = get_split_for_pert(gene_a, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get(batch_col, 'all') if batch_col else 'all'
        if batch not in control_bank:
            batch = list(control_bank.keys())[0] if control_bank else None
        if batch is None:
            continue
        
        clean_guide = guide_id.split(';')[0] if ';' in guide_id else guide_id
        seq_idx_a = dna_to_idx.get(clean_guide, dna_to_idx.get(guide_id, dna_to_idx.get(gene_a, -1)))
        
        ensg_a, ensg_b = guide_to_ensg.get(guide_id, (None, None))
        target_idx_a = gene_to_target_idx.get(ensg_a, gene_to_target_idx.get(gene_a, -1)) if ensg_a or gene_a else -1
        target_idx_b = gene_to_target_idx.get(ensg_b, gene_to_target_idx.get(gene_b, -1)) if ensg_b or gene_b else -1
        
        n_perts = 2 if gene_b else 1
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        
        if n_perts == 2:
            buf['seq_idx'].append([seq_idx_a, seq_idx_a, -1, -1])
            buf['target_idx'].append([target_idx_a, target_idx_b, -1, -1])
            buf['modality'].append([0, 0, -1, -1])
            buf['mode'].append([1, 1, -1, -1])
            buf['has_seq'].append([seq_idx_a >= 0, seq_idx_a >= 0, False, False])
            buf['has_target'].append([target_idx_a >= 0, target_idx_b >= 0, False, False])
        else:
            buf['seq_idx'].append([seq_idx_a, -1, -1, -1])
            buf['target_idx'].append([target_idx_a, -1, -1, -1])
            buf['modality'].append([0, -1, -1, -1])
            buf['mode'].append([1, -1, -1, -1])
            buf['has_seq'].append([seq_idx_a >= 0, False, False, False])
            buf['has_target'].append([target_idx_a >= 0, False, False, False])
        
        buf['n_perts'].append(n_perts)
        buf['dose'].append([-1.0, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_to_id.get('K562', 0))
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs, chunk_gene_counts
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val', 'test']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]}, test={pt_shard_counts["test"]} shards')
print(f'  Skipped {skipped_count} cells (missing embeddings), {low_gene_count} cells (low gene count), {unassigned_pert_count} cells (unassigned pert)')

norman: 100%|███████████████████████████████████████████████████████████████████████| 6/6 [02:17<00:00, 22.94s/it]


norman complete: train=0, val=0, test=0 shards
  Pretraining: train=35, val=3, test=7 shards
  Skipped 0 cells (missing embeddings), 0 cells (low gene count), 0 cells (unassigned pert)


In [34]:
del adata, control_bank
gc.collect()

0

## Process Sciplex Dataset (Chemical Perturbations)

In [35]:
ds_name = 'sciplex'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'  Shape: {adata.shape}')
print(f'  Obs columns: {adata.obs.columns.tolist()[:15]}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'  Genes in universe: {universe_mask.sum()}')

splits = dataset_splits.get(ds_name, {'train': [], 'val': [], 'test': []})
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

Processing sciplex...
  Shape: (799317, 110983)
  Obs columns: ['ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease']
  Genes in universe: 9974


In [36]:
pert_col = 'product_name' if 'product_name' in adata.obs.columns else 'perturbation'
dose_col = 'dose' if 'dose' in adata.obs.columns else None
cell_type_col = 'cell_type' if 'cell_type' in adata.obs.columns else None

control_bank, batch_col, condition_col = build_control_bank(
    adata, 'plate', pert_col, universe_mask, local_indices
)

  Built control bank with 52 batches, 17573 cells (filtered 5 low-gene-count)


In [37]:
target_col = 'target' if 'target' in adata.obs.columns else None
print(f'  Target column: {target_col}')

drug_to_target = {}
if target_col:
    for drug in adata.obs[pert_col].dropna().unique():
        if str(drug).lower() in ['control', 'vehicle', 'dmso', 'nan', '']:
            continue
        drug_rows = adata.obs[adata.obs[pert_col] == drug]
        targets = drug_rows[target_col].dropna().unique()
        if len(targets) > 0:
            target_name = str(targets[0]).strip()
            if target_name and target_name.lower() not in ['nan', '']:
                drug_to_target[drug] = target_name

print(f'  Drug -> target mappings: {len(drug_to_target)}')

  Target column: target
  Drug -> target mappings: 188


In [38]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': [], 'test': []}
pt_totals = {'train': [], 'val': [], 'test': []}
pt_shard_counts = {'train': 0, 'val': 0, 'test': 0}
skipped_count = 0
low_gene_count = 0
unassigned_pert_count = 0

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    chunk_gene_counts = (chunk_X > 0).sum(axis=1)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        if chunk_gene_counts[i] < MIN_GENES_PER_CELL:
            low_gene_count += 1
            continue
        
        drug = str(row.get(pert_col, '')).strip()
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = drug.lower() in ['control', 'vehicle', 'dmso', 'nan', '']
        
        if is_control:
            r = np.random.random()
            if r < TEST_PCT:
                pt_split = 'test'
            elif r < TEST_PCT + VAL_PCT:
                pt_split = 'val'
            else:
                pt_split = 'train'
        else:
            pt_split = get_split_for_pert(drug, train_perts, val_perts, test_perts)
        
        if pt_split is not None:
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        else:
            if not is_control:
                unassigned_pert_count += 1
        
        if is_control:
            continue
        
        if drug in sciplex_skip_drugs:
            skipped_count += 1
            continue
        
        split = get_split_for_pert(drug, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get(batch_col, 'all') if batch_col else 'all'
        if batch not in control_bank:
            batch = list(control_bank.keys())[0] if control_bank else None
        if batch is None:
            continue
        
        seq_idx = chemical_to_idx.get(drug, -1)
        target_name = drug_to_target.get(drug)
        target_idx = gene_to_target_idx.get(target_name, -1) if target_name else -1
        
        dose_val = float(row.get(dose_col, -1)) if dose_col else -1.0
        
        cell_type_name = str(row.get(cell_type_col, 'unknown')) if cell_type_col else 'unknown'
        cell_type_id = cell_type_to_id.get(cell_type_name, cell_type_to_id.get('unknown', 4))
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        buf['seq_idx'].append([seq_idx, -1, -1, -1])
        buf['target_idx'].append([target_idx, -1, -1, -1])
        buf['modality'].append([2, -1, -1, -1])
        buf['mode'].append([4, -1, -1, -1])
        buf['has_seq'].append([seq_idx >= 0, False, False, False])
        buf['has_target'].append([target_idx >= 0, False, False, False])
        buf['n_perts'].append(1)
        buf['dose'].append([dose_val, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_id)
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs, chunk_gene_counts
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val', 'test']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]}, test={pt_shard_counts["test"]} shards')
print(f'  Skipped {skipped_count} cells (missing embeddings), {low_gene_count} cells (low gene count), {unassigned_pert_count} cells (unassigned pert)')

sciplex: 100%|████████████████████████████████████████████████████████████████████| 40/40 [35:31<00:00, 53.28s/it]


sciplex complete: train=248, val=15, test=30 shards
  Pretraining: train=266, val=16, test=32 shards
  Skipped 0 cells (missing embeddings), 163 cells (low gene count), 0 cells (unassigned pert)


In [39]:
del adata, control_bank
gc.collect()

0

## Verify Alignment Pairs (from Notebook 2)

Alignment pairs are now generated in Notebook 2, not here. We just verify they exist.

In [40]:
print('Alignment pairs (from Notebook 2):')
for split in ['train', 'val', 'test']:
    align_file = pert_embd_dir / split / f'align_{split}.npz'
    if align_file.exists():
        with np.load(align_file) as data:
            print(f'  {split}: {len(data["seq_idx"])} pairs')
    else:
        print(f'  {split}: NOT FOUND - run Notebook 2 first')

input_to_id_path = pert_embd_dir / 'input_to_id.json'
if input_to_id_path.exists():
    with open(input_to_id_path) as f:
        input_to_id = json.load(f)
    print(f'\ninput_to_id.json: {len(input_to_id)} entries')
    crispri_count = sum(1 for k in input_to_id if '_crispri_' in k)
    crispra_count = sum(1 for k in input_to_id if '_crispra_' in k)
    inhibitor_count = sum(1 for k in input_to_id if '_inhibitor_' in k)
    print(f'  CRISPRi: {crispri_count}, CRISPRa: {crispra_count}, Inhibitor: {inhibitor_count}')
else:
    print(f'\nWarning: input_to_id.json not found - run Notebook 2 first')

Alignment pairs (from Notebook 2):
  train: 10791 pairs
  val: 107 pairs
  test: 329 pairs

input_to_id.json: 30178 entries
  CRISPRi: 29701, CRISPRa: 289, Inhibitor: 188


## Summary and Validation

In [41]:
print('=== Data Prep Notebook 3 Complete ===')
print('\nTraining shards:')
for split in ['train', 'val', 'test']:
    shards = list((training_dir / split).glob('shard_*.npz'))
    print(f'  {split}: {len(shards)} shards')

print('\nPretraining shards:')
for split in ['train', 'val', 'test']:
    shards = list((pretraining_dir / split).glob('pt_*.npz'))
    print(f'  {split}: {len(shards)} shards')

=== Data Prep Notebook 3 Complete ===

Training shards:
  train: 1035 shards
  val: 68 shards
  test: 133 shards

Pretraining shards:
  train: 1114 shards
  val: 73 shards
  test: 145 shards


In [42]:
print('\nValidation - checking first training shard:')
train_shards = list((training_dir / 'train').glob('shard_*.npz'))
if train_shards:
    with np.load(train_shards[0]) as data:
        print(f'  control shape: {data["control"].shape}')
        print(f'  case shape: {data["case"].shape}')
        print(f'  seq_idx shape: {data["seq_idx"].shape}')
        print(f'  target_idx shape: {data["target_idx"].shape}')
        print(f'  n_perts range: {data["n_perts"].min()} - {data["n_perts"].max()}')
        print(f'  gene_mask sum: {data["gene_mask"].sum()}')


Validation - checking first training shard:
  control shape: (2560, 10000)
  case shape: (2560, 10000)
  seq_idx shape: (2560, 4)
  target_idx shape: (2560, 4)
  n_perts range: 1 - 1
  gene_mask sum: 8248
